# Eksploracja danych  - PS 3 - Tomasz Jarmoc i Krzysztof Sobolewski


Dla zbioru danych użytego w raporcie 3 (do klasyfikacji) przeprowadź analizę skupień wyłączając kolumnę ze zmienną zależną (lub etykietami klasy, jeśli to zbiór do klasyfikacji). W drugim przypadku użyj kolumny z etykietami klasy do sprawdzenia jakości uzyskanych klastrów.

Użyj:

- metod hierarchicznych (z 3 różnymi sposobami liczenia odległości między klastrami) - sporządź dendrogram dla każdej z metod
- metody k-średnich
- metody EM
- metody DBSCAN
- jednej innej - dowolnej metody

Porównaj uzyskane wyniki - liczbę uzyskanych klastrów i ich jakość.
Dla tego zbioru zmienna zależna to `charges`, więc nie bierzemy jej do grupowania. Ponieważ jest ciągła, porównujemy klastry z `charges` przez R2 liczone na podstawie średniej wartości w klastrze; nie stosujemy ARI ani NMI.


In [1]:
import os
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering, Birch, DBSCAN, KMeans
from sklearn.compose import ColumnTransformer
from sklearn.mixture import GaussianMixture
from sklearn.metrics import r2_score, silhouette_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler

plik_csv = 'insurance.csv'
link_do_pliku = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'
if not os.path.exists(plik_csv):
    urlretrieve(link_do_pliku, plik_csv)

df = pd.read_csv(plik_csv)
print(f'Wymiary danych: {df.shape}')
display(df.head())


Wymiary danych: (1338, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [ ]:
# Zmienna zalezna charges jest wylaczona z cech klasteryzacji.
y = df['charges']
X = df.drop(columns='charges')
kolumny_numeryczne = X.select_dtypes(include=np.number).columns.tolist()
kolumny_kategoryczne = X.select_dtypes(exclude=np.number).columns.tolist()

przetwarzanie = ColumnTransformer(transformers=[
    ('num', StandardScaler(), kolumny_numeryczne),
    ('kat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), kolumny_kategoryczne),
])
X_scaled = przetwarzanie.fit_transform(X)
print(f'Liczba cech po kodowaniu: {X_scaled.shape[1]}')


## 1. Metody hierarchiczne

Ponizej porownujemy dendrogramy dla metod Ward, complete i average. Dla czytelnosci pokazane jest ostatnie 30 polaczen.

In [ ]:
metody_linkage = ['ward', 'complete', 'average']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metoda in zip(axes, metody_linkage):
    Z = linkage(X_scaled, method=metoda, metric='euclidean')
    dendrogram(Z, truncate_mode='lastp', p=30, ax=ax)
    ax.set_title(f'Dendrogram: {metoda}')
    ax.set_xlabel('Klastry (skrocony dendrogram)')
    ax.set_ylabel('Odleglosc')

plt.tight_layout()
display(fig)
plt.close(fig)


## 2. Pozostale metody klasteryzacji

Dla porownywalnosci ustalamy k=2 w metodach wymagajacych liczby klastrow. DBSCAN sam wyznacza klastry; etykieta -1 oznacza szum.

In [ ]:
wyniki = {}

for metoda in metody_linkage:
    model = AgglomerativeClustering(n_clusters=2, linkage=metoda)
    wyniki[f'HAC ({metoda})'] = model.fit_predict(X_scaled)

wyniki['K-srednich'] = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_scaled)
wyniki['EM (Gaussian Mixture)'] = GaussianMixture(n_components=2, random_state=42).fit_predict(X_scaled)
wyniki['DBSCAN'] = DBSCAN(eps=1.0, min_samples=10).fit_predict(X_scaled)
wyniki['Birch'] = Birch(n_clusters=2).fit_predict(X_scaled)

print(f'Uruchomiono {len(wyniki)} wariantow klasteryzacji.')


## 3. Porownanie jakosci

Silhouette ocenia spojnosc i separacje klastrow. R2 pokazuje, jaka czesc zroznicowania `charges` jest opisana przez srednie wartosci tej zmiennej w klastrach. Dla DBSCAN obserwacje szumowe sa pomijane w obu metrykach.


In [ ]:
wiersze = []
for nazwa, etykiety in wyniki.items():
    maska = etykiety != -1
    etykiety_bez_szumu = etykiety[maska]
    klastry = np.unique(etykiety_bez_szumu)

    silhouette = np.nan
    if len(klastry) >= 2 and maska.sum() > len(klastry):
        silhouette = silhouette_score(
            X_scaled[maska], etykiety_bez_szumu,
            sample_size=min(1000, int(maska.sum())), random_state=42,
        )

    charges_r2 = np.nan
    if maska.sum() > 0:
        charges_w_klastrach = y.to_numpy()[maska]
        srednie_charges = pd.Series(charges_w_klastrach).groupby(etykiety_bez_szumu).transform('mean')
        charges_r2 = r2_score(charges_w_klastrach, srednie_charges)

    wiersze.append({
        'Metoda': nazwa,
        'Liczba klastrow': len(klastry),
        'Szum (DBSCAN)': int((etykiety == -1).sum()),
        'Silhouette': silhouette,
        'R2 wzgledem charges': charges_r2,
    })

porownanie = pd.DataFrame(wiersze).set_index('Metoda')
display(porownanie.round(3))


## 4. Wykresy porownawcze

Wykresy pokazuja liczbe klastrow, Silhouette oraz R2 wzgledem zmiennej zaleznej `charges`.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

porownanie['Liczba klastrow'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Liczba klastrow')
axes[0].set_ylabel('Liczba')

porownanie['Silhouette'].dropna().plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title('Silhouette (wyzej = lepiej)')

porownanie['R2 wzgledem charges'].plot(kind='bar', ax=axes[2], color='seagreen')
axes[2].set_title('R2 wzgledem charges (wyzej = lepiej)')
axes[2].set_ylabel('R2')

for ax in axes:
    ax.tick_params(axis='x', labelrotation=60)
    ax.grid(axis='y', alpha=0.25)

plt.tight_layout()
display(fig)
plt.close(fig)
